# M-Lab India — clean data exploration

This notebook only looks at the **cleaned** dataset produced by `outlier.ipynb`
(data-centre/enterprise traffic and the sentinel RTT value already removed). No outlier-detection
code lives here — for the removal evidence and reasoning, see the inspection notebook.

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, StrMethodFormatter
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'mlab_analysis':
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = Path(os.environ.get('MLAB_OUTPUT_DIR', PROJECT_ROOT / 'outputs' / 'analysis' / '2026-06-18_to_2026-11-04'))
FIGURE_DIR = DATA_DIR / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

def figure_path(filename):
    path = FIGURE_DIR / '03_clean_data' / filename
    path.parent.mkdir(parents=True, exist_ok=True)
    return path

BLUE, ORANGE, GREY = '#4C78A8', '#F58518', '#B0B0B0'

download = pd.read_csv(DATA_DIR / 'download_clean.csv', low_memory=False)
upload = pd.read_csv(DATA_DIR / 'upload_clean.csv', low_memory=False)
DOWNLOAD_FEATURES = ['download_mbps', 'min_rtt_ms', 'loss_rate']
UPLOAD_FEATURES = ['upload_mbps', 'min_rtt_ms']

print(f'Clean download: {download.shape}   Clean upload: {upload.shape}')

## 1. Coverage

Measurements per day over the observed window, on the clean data.

In [ ]:
all_dates = pd.concat([pd.to_datetime(download['date']), pd.to_datetime(upload['date'])])
expected_dates = pd.date_range(all_dates.min(), all_dates.max(), freq='D')

fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
for ax, (name, df), color in zip(axes, [('download', download), ('upload', upload)], [BLUE, ORANGE]):
    counts = pd.to_datetime(df['date']).value_counts().reindex(expected_dates, fill_value=0).sort_index()
    observed = int((counts > 0).sum())
    mean_daily = counts.mean()
    median_daily = counts.median()
    peak_daily = int(counts.max())
    ax.bar(counts.index, counts.values, width=.85, color=color)
    ax.axhline(mean_daily, color='black', linestyle='--', linewidth=1,
               label=f'Mean {mean_daily:,.1f}/day')
    ax.set_title(f'{name.title()}: {observed}/{len(expected_dates)} days; n={len(df):,}; '
                 f'median={median_daily:,.0f}/day; peak={peak_daily:,}/day')
    ax.set_ylabel('Measurements per day'); ax.yaxis.set_major_formatter(StrMethodFormatter('{x:,.0f}'))
    ax.grid(axis='y', alpha=.25); ax.legend(loc='upper right', fontsize=8)
axes[-1].set_xlabel(f'{expected_dates.min():%Y-%m-%d} to {expected_dates.max():%Y-%m-%d}')
fig.tight_layout(); fig.savefig(figure_path('coverage.png'), dpi=180); plt.show()

## 2. Distributions & ECDFs

Log-scaled x-axes for throughput/RTT; loss stays linear on its natural `[0, 1]` scale.

In [ ]:
def smooth_histogram(density, sigma=1.5):
    radius = max(2, int(4 * sigma))
    x = np.arange(-radius, radius + 1)
    kernel = np.exp(-0.5 * (x / sigma) ** 2)
    kernel /= kernel.sum()
    return np.convolve(density, kernel, mode='same')

def distribution_ecdf(df, dataset, features, color):
    fig = plt.figure(figsize=(14, 4.1 * len(features)))
    grid = fig.add_gridspec(len(features) * 2, 2, height_ratios=[1, .16] * len(features),
                            hspace=.65, wspace=.22)
    axes = np.empty((len(features), 2), dtype=object)
    for row, feature in enumerate(features):
        axes[row, 0] = fig.add_subplot(grid[row * 2, 0])
        axes[row, 1] = fig.add_subplot(grid[row * 2, 1])
        summary_ax = fig.add_subplot(grid[row * 2 + 1, :])
        summary_ax.set_axis_off()
        values = df[feature].dropna().to_numpy()
        is_loss = 'loss' in feature
        display_feature = {'download_mbps': 'Download throughput', 'upload_mbps': 'Upload throughput',
                           'min_rtt_ms': 'Minimum RTT', 'loss_rate': 'Packet loss'}[feature]
        if is_loss:
            values = values * 100
        unit = '%' if is_loss else ('Mbps' if 'mbps' in feature else 'ms')
        percentiles = (50, 75, 80, 85, 90, 95, 99) if unit == 'ms' or is_loss else (1, 5, 10, 15, 20, 25, 50)
        percentile_values = np.percentile(values, percentiles)
        mean, median, std = np.mean(values), np.median(values), np.std(values, ddof=1)
        bins = np.linspace(0, values.max(), 51) if is_loss else np.geomspace(values.min(), values.max(), 51)
        counts, edges, _ = axes[row, 0].hist(values, bins=bins, color=color, alpha=.55, label='Observations per bin')
        centers = (edges[:-1] + edges[1:]) / 2 if is_loss else np.sqrt(edges[:-1] * edges[1:])
        if not is_loss:
            axes[row, 0].plot(centers, smooth_histogram(counts), color='#E45756', lw=2.2, label='Smoothed bin counts')
            axes[row, 0].set_xscale('log'); axes[row, 1].set_xscale('log')
        ordered = np.sort(values)
        cumulative_pct = np.arange(1, len(ordered) + 1) / len(ordered) * 100
        axes[row, 1].plot(ordered, cumulative_pct, color='#F2B701' if color == BLUE else '#8C6D31')
        axes[row, 0].axvline(median, color='black', ls='--', lw=1.2, label=f'Median={median:.4g} {unit}')
        axes[row, 1].axvline(median, color='black', ls='--', lw=1.2, label=f'Median={median:.4g} {unit}')
        axes[row, 1].scatter(percentile_values, percentiles, s=14, color='#E45756', zorder=3,
                             label='Reported percentile points')
        axes[row, 0].set(title=f'{dataset.title()}: {display_feature} distribution (n={len(values):,})',
                         xlabel=f'{display_feature} ({unit})', ylabel='Measurements per bin')
        axes[row, 1].set(title=f'{dataset.title()}: {display_feature} ECDF (n={len(values):,})',
                         xlabel=f'{display_feature} ({unit})', ylabel='Measurements at or below (%)', ylim=(0, 101))
        if not is_loss:
            for ax in axes[row]:
                ax.xaxis.set_major_formatter(FuncFormatter(lambda value, _: f'{value:g}'))
        axes[row, 0].yaxis.set_major_formatter(StrMethodFormatter('{x:,.0f}'))
        axes[row, 1].yaxis.set_major_formatter(FuncFormatter(lambda value, _: f'{value:.0f}%'))
        stats_lines = [f'n={len(values):,}   mean={mean:.4g} {unit}',
                       f'median={median:.4g} {unit}   SD={std:.4g} {unit}']
        stats_lines.extend('   '.join(f'P{p}={v:.4g}' for p, v in zip(percentiles[i:i + 3], percentile_values[i:i + 3]))
                           for i in range(0, len(percentiles), 3))
        summary_text = '   '.join(stats_lines[:2]) + '\nPercentiles: ' + '   '.join(
            f'P{p}={v:.4g} {unit}' for p, v in zip(percentiles, percentile_values))
        summary_ax.text(.5, .5, summary_text,
                        ha='center', va='center', fontsize=8,
                        bbox=dict(facecolor='white', edgecolor='0.8', boxstyle='round,pad=.4'))
        axes[row, 0].legend(fontsize=7.5); axes[row, 1].legend(fontsize=7.5, loc='lower right')
        axes[row, 0].grid(alpha=.25); axes[row, 1].grid(alpha=.25)
    fig.tight_layout(); fig.savefig(figure_path(f'{dataset}_distributions_ecdf.png'), dpi=180, bbox_inches='tight'); plt.show()

distribution_ecdf(download, 'download', DOWNLOAD_FEATURES, BLUE)
distribution_ecdf(upload, 'upload', UPLOAD_FEATURES, ORANGE)

## 3. Geo-spatial analysis

Density and throughput maps for the clean data, plus median throughput/RTT by state.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharex=True, sharey=True)
for ax, (name, df) in zip(axes, [('download', download), ('upload', upload)]):
    mesh = ax.hexbin(df['longitude'], df['latitude'], gridsize=38, mincnt=1,
                     norm=plt.matplotlib.colors.LogNorm(vmin=1), cmap='viridis')
    occupied_hexes = len(mesh.get_array())
    ax.set(xlim=(67, 99), ylim=(6, 38),
           title=f'{name.title()}: measurement density (n={len(df):,})', xlabel='Longitude', ylabel='Latitude')
    ax.text(.02, .98, f'Occupied hexagons: {occupied_hexes:,}', transform=ax.transAxes,
            va='top', fontsize=8, bbox=dict(facecolor='white', alpha=.85, edgecolor='none'))
    ax.grid(alpha=.15)
    cb = fig.colorbar(mesh, ax=ax, label='Measurements per hexagon (log scale)')
    cb.ax.yaxis.set_major_formatter(FuncFormatter(lambda value, _: f'{value:,.0f}'))
fig.tight_layout(); fig.savefig(figure_path('geography_density.png'), dpi=200); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharex=True, sharey=True)
for ax, (name, df, col) in zip(axes, [('download', download, 'download_mbps'), ('upload', upload, 'upload_mbps')]):
    sc = ax.scatter(df['longitude'], df['latitude'], c=df[col], cmap='inferno',
                     norm=plt.matplotlib.colors.LogNorm(), s=4, alpha=.5)
    ax.set(xlim=(67, 99), ylim=(6, 38), title=f'{name.title()}: throughput by location (clean)', xlabel='Longitude', ylabel='Latitude')
    median_speed = df[col].median()
    ax.text(.02, .98, f'n={len(df):,}   median={median_speed:.3g} Mbps', transform=ax.transAxes,
            va='top', fontsize=8, color='white',
            bbox=dict(facecolor='black', alpha=.65, edgecolor='none'))
    ax.grid(alpha=.15)
    cb = fig.colorbar(sc, ax=ax, label=f'{col} (Mbps, log scale)')
    cb.ax.yaxis.set_major_formatter(FuncFormatter(lambda value, _: f'{value:g}'))
fig.tight_layout(); fig.savefig(figure_path('geography_throughput.png'), dpi=200); plt.show()

top_states = download['state'].value_counts().head(12).index
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
for ax, (name, df, col) in zip(axes, [('download', download, 'download_mbps'), ('upload', upload, 'upload_mbps')]):
    state_data = df[df['state'].isin(top_states)]
    med = state_data.groupby('state')[col].median().sort_values()
    state_n = state_data.groupby('state')[col].count().reindex(med.index)
    bars = ax.barh(med.index, med.values, color=BLUE if name == 'download' else ORANGE)
    ax.bar_label(bars, labels=[f'{value:.2f}' for value in med.values], padding=3, fontsize=8)
    ax.set_yticks(range(len(med)))
    ax.set_yticklabels([f'{state} (n={state_n[state]:,})' for state in med.index])
    ax.set_xlim(0, med.max() * 1.18)
    ax.set(title=f'{name.title()}: median throughput by state (n={len(state_data):,})',
           xlabel='Median throughput (Mbps)')
    ax.grid(axis='x', alpha=.25)
fig.tight_layout(); fig.savefig(figure_path('geography_state_median.png'), dpi=200, bbox_inches='tight'); plt.show()